## Checking

In [1]:
!nvidia-smi

Tue Jul 21 19:54:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Setup

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
!mkdir -p /content/drive/MyDrive/smallnet_colab_backup

In [3]:
%cd /content
!rm -rf smallnet
!git clone https://github.com/SepehrAkbari/smallnet.git
%cd /content/smallnet

/content
Cloning into 'smallnet'...
remote: Enumerating objects: 1954, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 1954 (delta 27), reused 46 (delta 19), pack-reused 1885 (from 1)
Receiving objects: 100% (1954/1954), 580.96 MiB | 19.88 MiB/s, done.
Resolving deltas: 100% (262/262), done.
Updating files: 100% (1605/1605), done.
/content/smallnet


In [10]:
!find /content/drive/MyDrive/model -maxdepth 1 -type f -print

In [4]:
!mkdir -p /content/smallnet/model
!cp -v /content/drive/MyDrive/model/* /content/smallnet/model/

'/content/drive/MyDrive/model/best_model.pth' -> '/content/smallnet/model/best_model.pth'
'/content/drive/MyDrive/model/finetuned_rank_128.pth' -> '/content/smallnet/model/finetuned_rank_128.pth'
'/content/drive/MyDrive/model/finetuned_rank_256.pth' -> '/content/smallnet/model/finetuned_rank_256.pth'
'/content/drive/MyDrive/model/finetuned_rank_64.pth' -> '/content/smallnet/model/finetuned_rank_64.pth'


In [5]:
!git log -1 --oneline

5f68c34 (HEAD -> main, origin/main, origin/HEAD) .


In [14]:
!ls configs/camvid_vgg_cp_paper.json
!ls scripts/run_experiment.py
!ls model/best_model.pth
!ls data/CamVid/class_dict.csv
!ls data/CamVid/train | head
!ls data/CamVid/test_labels | head

In [6]:
%cd /content/smallnet
!pwd
!ls

/content/smallnet
/content/smallnet
AGENTS.md  data  LICENSE  notebook	  README.md  results  src    uv.lock
configs    docs  model	  pyproject.toml  res	     scripts  tests


In [7]:
!pip install -q uv
!uv sync
!uv run python -m pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 33.8 MB/s eta 0:00:00:00:0100:01
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 1ms
Prepared 79 packages in 35.65s                                           
Installed 79 packages in 255ms                              
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.3.1
 + executing==2.2.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jinja2==3.1.6
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + kiwisolver==1.5.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mpmath==1.3.0
 + nest-asyncio==1.6.0
 + networkx==3.6.1
 + numpy=

In [17]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [18]:
import torch

checkpoint = torch.load(
    "/content/smallnet/model/best_model.pth",
    map_location="cpu"
)

print("Checkpoint loaded.")
print("Number of state-dict entries:", len(checkpoint))

In [19]:
!mkdir -p results/camvid_vgg_cp results/paper
!cp -r /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/. \
    results/camvid_vgg_cp/ 2>/dev/null || true
!cp -r /content/drive/MyDrive/smallnet_colab_backup/paper/. \
    results/paper/ 2>/dev/null || true

In [20]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage validate-data \
  --device cpu

In [21]:
import json

with open("results/camvid_vgg_cp/dataset_validation_report.json") as f:
    report = json.load(f)

print("Status:", report["status"])
print("Mapped unknown pixels:", report.get("unknown_pixels_mapped_to_ignore"))

## rank 32

In [22]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 32 \
  --seeds 0

In [23]:
import pandas as pd

df = pd.read_csv(
    "results/camvid_vgg_cp/reconstruction_summary.csv"
)
display(df.tail())

In [24]:
failed = df[
    df.astype(str)
      .apply(lambda col: col.str.contains("fail|error", case=False, na=False))
      .any(axis=1)
]

display(failed)

In [25]:
!rm -rf /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp
!rm -rf /content/drive/MyDrive/smallnet_colab_backup/paper

!cp -r results/camvid_vgg_cp \
    /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp

!cp -r results/paper \
    /content/drive/MyDrive/smallnet_colab_backup/paper

In [26]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 32 \
  --seeds 1 2

In [27]:
import pandas as pd
from pathlib import Path

summary_path = Path(
    "/content/smallnet/results/camvid_vgg_cp/reconstruction_summary.csv"
)

df = pd.read_csv(summary_path)

print("Number of rows:", len(df))
print("\nRank values and Python types after CSV loading:")
for value in df["rank"].unique():
    print(repr(value), type(value))

display(
    df[
        [
            "method",
            "rank",
            "seed",
            "status",
            "actual_relative_squared_frobenius_error",
        ]
    ]
)

In [28]:
rank_numeric = pd.to_numeric(df["rank"], errors="coerce")

display(
    df[rank_numeric == 32][
        [
            "method",
            "rank",
            "seed",
            "status",
            "actual_relative_squared_frobenius_error",
            "decomposition_runtime_seconds",
        ]
    ]
)

In [29]:
!mkdir -p /content/drive/MyDrive/smallnet_colab_backup

!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [35]:
%cd /content/smallnet
!git status --short

In [36]:
!git pull origin main

In [37]:
!uv sync
!uv run python -m pytest

In [39]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 32 \
  --seeds 0 1 2

In [40]:
import pandas as pd

df = pd.read_csv(
    "results/camvid_vgg_cp/reconstruction_summary.csv"
)

df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
df["seed"] = pd.to_numeric(df["seed"], errors="coerce")

rank32 = df[df["rank"] == 32]

display(
    rank32[
        [
            "method",
            "rank",
            "seed",
            "status",
            "actual_relative_squared_frobenius_error",
            "decomposition_runtime_seconds",
        ]
    ].sort_values(["method", "seed"])
)

completed_seeds = sorted(
    rank32.loc[
        (rank32["method"] == "cp")
        & (rank32["status"] == "completed"),
        "seed",
    ]
    .dropna()
    .astype(int)
    .unique()
)

print("Completed CP seeds:", completed_seeds)

In [41]:
!ls -lh results/paper/figures/figure_b_reconstruction_squared_error.*

In [42]:
import json

with open("results/camvid_vgg_cp/reconstruction_metadata.json") as f:
    metadata = json.load(f)

print("Failures:", metadata.get("failures"))
print("Figure outputs:", metadata.get("paper_figure_outputs"))
print("Figure failures:", metadata.get("figure_failures"))

In [43]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

## rank 64

In [44]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 64 \
  --seeds 0 1 2

In [45]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

## rank 128

In [46]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 128 \
  --seeds 0 1 2

In [47]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

## rank 256

In [48]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 256 \
  --seeds 0 1 2

In [49]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

## rank 512

In [50]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage reconstruction \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2

In [51]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

## Validation

In [52]:
import pandas as pd

df = pd.read_csv(
    "results/camvid_vgg_cp/reconstruction_summary.csv"
)

df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
df["seed"] = pd.to_numeric(df["seed"], errors="coerce")

display(
    df[
        [
            "method",
            "rank",
            "seed",
            "status",
            "actual_relative_squared_frobenius_error",
            "decomposition_runtime_seconds",
        ]
    ].sort_values(["rank", "method", "seed"])
)

In [53]:
expected_ranks = {32, 64, 128, 256, 512}

cp_completed = df[
    (df["method"] == "cp")
    & (df["status"] == "completed")
]

svd_completed = df[
    (df["method"] == "matrix_svd_output_unfolding")
    & (df["status"] == "completed")
]

cp_counts = (
    cp_completed.groupby("rank")["seed"]
    .nunique()
    .to_dict()
)

svd_ranks = set(
    svd_completed["rank"]
    .dropna()
    .astype(int)
)

print("CP completed seeds per rank:", cp_counts)
print("Matrix-SVD completed ranks:", sorted(svd_ranks))

assert all(cp_counts.get(rank, 0) == 3 for rank in expected_ranks)
assert svd_ranks == expected_ranks

print("Reconstruction sweep is complete.")

In [54]:
failed = df[df["status"] != "completed"]
display(failed)

In [55]:
!ls -lh results/paper/figures/figure_a_unfolding_cumulative_energy.*
!ls -lh results/paper/figures/figure_b_reconstruction_squared_error.*

In [56]:
rank_summary = pd.read_csv(
    "results/camvid_vgg_cp/reconstruction_rank_summary.csv"
)

display(rank_summary.sort_values("rank"))

## Zero-shot

In [57]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 32 \
  --seeds 0 1 2

In [59]:
from pathlib import Path
import pandas as pd

for path in sorted(Path("results/camvid_vgg_cp").glob("*structural*csv")):
    print("\nFILE:", path)
    table = pd.read_csv(path)
    print("Rows:", len(table))
    display(table)

In [60]:
import json

with open(
    "results/camvid_vgg_cp/structural_zero_shot_metadata.json"
) as f:
    metadata = json.load(f)

print("Failures:", metadata.get("failures"))
print("Execution ranks:", metadata.get("execution_ranks"))
print("Execution seeds:", metadata.get("execution_cp_seeds"))
print("Correlation results:", metadata.get("exploratory_correlations"))

In [61]:
summary = pd.read_csv(
    "results/camvid_vgg_cp/structural_zero_shot_summary.csv"
)

summary["rank"] = pd.to_numeric(summary["rank"], errors="coerce")
summary["seed"] = pd.to_numeric(summary["seed"], errors="coerce")

rank32 = summary[summary["rank"] == 32]

display(
    rank32.sort_values(
        ["method", "seed", "split"]
    )
)

In [62]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [63]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 64 \
  --seeds 0 1 2

In [1]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [2]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 128 \
  --seeds 0 1 2

In [ ]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [ ]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 256 \
  --seeds 0 1 2

In [ ]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [ ]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2

In [ ]:
!rsync -a /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [ ]:
from pathlib import Path

for path in sorted(Path("results/camvid_vgg_cp").glob("*structural*summary.csv")):
    print("\nFILE:", path)
    table = pd.read_csv(path)
    print("Rows:", len(table))
    display(table.head())

In [ ]:
!find results/paper/figures -maxdepth 1 -type f | sort

## v2

In [8]:
!mkdir -p /content/smallnet/results/camvid_vgg_cp
!mkdir -p /content/smallnet/results/paper

!rsync -a \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/ \
  /content/smallnet/results/camvid_vgg_cp/

!rsync -a \
  /content/drive/MyDrive/smallnet_colab_backup/paper/ \
  /content/smallnet/results/paper/

In [9]:
!find results/camvid_vgg_cp -maxdepth 1 -type f | sort

results/camvid_vgg_cp/cp_finetune_config_used.json
results/camvid_vgg_cp/cp_zero_shot_config_used.json
results/camvid_vgg_cp/cp_zero_shot_metadata.json
results/camvid_vgg_cp/cp_zero_shot_summary.csv
results/camvid_vgg_cp/dataset_class_counts.csv
results/camvid_vgg_cp/dataset_mask_forensics.json
results/camvid_vgg_cp/dataset_unknown_colors_by_file.csv
results/camvid_vgg_cp/dataset_validation_config_used.json
results/camvid_vgg_cp/dataset_validation_report.json
results/camvid_vgg_cp/dataset_validation_summary.csv
results/camvid_vgg_cp/dense_eval_config_used.json
results/camvid_vgg_cp/dense_eval_metadata.json
results/camvid_vgg_cp/dense_eval_summary.csv
results/camvid_vgg_cp/existing_finetuned_config_used.json
results/camvid_vgg_cp/existing_finetuned_metadata.json
results/camvid_vgg_cp/existing_finetuned_summary.csv
results/camvid_vgg_cp/profile_config_used.json
results/camvid_vgg_cp/profile_metadata.json
results/camvid_vgg_cp/profile_summary.csv
results/camvid_vgg_cp/rank_diagnostics_con

In [10]:
import pandas as pd

recon = pd.read_csv(
    "results/camvid_vgg_cp/reconstruction_summary.csv"
)

recon["rank"] = pd.to_numeric(recon["rank"], errors="coerce")
recon["seed"] = pd.to_numeric(recon["seed"], errors="coerce")

cp_recon = recon[
    (recon["method"] == "cp")
    & (recon["status"] == "completed")
]

print(
    cp_recon.groupby("rank")["seed"]
    .nunique()
    .sort_index()
)

rank
32     3
64     3
128    3
256    3
512    3
Name: seed, dtype: int64


In [11]:
from pathlib import Path
import pandas as pd

zero_path = Path(
    "results/camvid_vgg_cp/structural_zero_shot_summary.csv"
)

if zero_path.exists():
    zero = pd.read_csv(zero_path)
    zero["rank"] = pd.to_numeric(zero["rank"], errors="coerce")
    zero["seed"] = pd.to_numeric(zero["seed"], errors="coerce")

    print("Rows restored:", len(zero))
    display(
        zero.sort_values(
            ["rank", "method", "seed", "split"]
        )
    )
else:
    print("No structural zero-shot summary was present in the Drive backup.")

Rows restored: 18


,actual_relative_frobenius_error,actual_relative_squared_frobenius_error,all_class_miou,checkpoint_sha256,dataset_validation_report_reference,dataset_validation_report_sha256,dense_target_layer_parameter_count,frequency_weighted_iou,full_model_macs,full_model_parameter_count,...,model_kind,output_mode_tail_bound_squared,pixel_accuracy,present_class_miou,rank,seed,split,status,target_layer_macs,target_layer_parameter_count
5,0.975077,0.950774,0.186272,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.705828,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.825552,0.199118,32.0,0.0,test,completed,24409440,152032
4,0.975077,0.950774,0.204940,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.728818,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.843595,0.219074,32.0,0.0,val,completed,24409440,152032
7,0.975021,0.950666,0.169497,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.694370,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.819407,0.181186,32.0,1.0,test,completed,24409440,152032
6,0.975021,0.950666,0.193359,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.717831,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.836733,0.206694,32.0,1.0,val,completed,24409440,152032
9,0.975070,0.950762,0.180743,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.696061,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.818746,0.193209,32.0,2.0,test,completed,24409440,152032
8,0.975070,0.950762,0.198300,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.714238,54491706720,31779136,...,cp_structural_zero_shot,0.822257,0.833291,0.211976,32.0,2.0,val,completed,24409440,152032
3,0.906784,0.822257,0.279448,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.756583,54621388800,32565088,...,matrix_svd_output_unfolding_zero_shot,0.822257,0.853491,0.298720,32.0,NaN,test,completed,154091520,937984
2,0.906784,0.822257,0.303934,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.781821,54621388800,32565088,...,matrix_svd_output_unfolding_zero_shot,0.822257,0.871473,0.324895,32.0,NaN,val,completed,154091520,937984
13,0.964918,0.931066,0.215574,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.728018,54516116160,31927072,...,cp_structural_zero_shot,0.781806,0.838502,0.230441,64.0,0.0,test,completed,48818880,299968
12,0.964918,0.931066,0.231327,1576b92ebb359a519d57d489f8ee823c033588fbd7c77b...,/content/smallnet/results/camvid_vgg_cp/datase...,f1bb932fa776a2065507de6b29bdc174c139a1e20f8c41...,102764544,0.749633,54516116160,31927072,...,cp_structural_zero_shot,0.781806,0.855502,0.247280,64.0,0.0,val,completed,48818880,299968


In [12]:
if zero_path.exists():
    cp_zero = zero[
        (zero["method"] == "cp")
        & (zero["status"] == "completed")
    ]

    completion = (
        cp_zero.groupby(["rank", "seed"])["split"]
        .apply(lambda values: sorted(set(values)))
        .reset_index(name="completed_splits")
    )

    display(completion)

,rank,seed,completed_splits
0,32.0,0.0,"[test, val]"
1,32.0,1.0,"[test, val]"
2,32.0,2.0,"[test, val]"
3,64.0,0.0,"[test, val]"
4,64.0,1.0,"[test, val]"
5,64.0,2.0,"[test, val]"


In [13]:
expected_ranks = [32, 64, 128, 256, 512]
expected_seeds = [0, 1, 2]

completed_pairs = set()

if zero_path.exists():
    cp_completed = zero[
        (zero["method"] == "cp")
        & (zero["status"] == "completed")
    ]

    for (rank, seed), group in cp_completed.groupby(["rank", "seed"]):
        splits = set(group["split"].astype(str))
        if len(splits) >= 2:
            completed_pairs.add((int(rank), int(seed)))

missing_by_rank = {}

for rank in expected_ranks:
    missing = [
        seed
        for seed in expected_seeds
        if (rank, seed) not in completed_pairs
    ]
    if missing:
        missing_by_rank[rank] = missing

print("Missing CP zero-shot runs:")
print(missing_by_rank)

Missing CP zero-shot runs:
{128: [0, 1, 2], 256: [0, 1, 2], 512: [0, 1, 2]}


In [14]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 128 \
  --seeds 0 1 2
  
!rsync -a \
  /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a \
  /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

Wrote:
  /content/smallnet/results/camvid_vgg_cp/structural_zero_shot_metadata.json


In [15]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 256 \
  --seeds 0 1 2
  
!rsync -a \
  /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a \
  /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

Wrote:
  /content/smallnet/results/camvid_vgg_cp/structural_zero_shot_metadata.json


In [ ]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage structural-zero-shot \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2
  
!rsync -a \
  /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a \
  /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/